# Data Preprocessing Notes

## Changes Made

- Manually removed the initial and end headers
- Removed the Project Gutenberg license and Table of Contents

## Paragraph Segmentation

Performed paragraph segmentation to ensure the model is not trained on correlations between text length and authenticity (long text = human, short text = AI).

## Capitalization and Punctuation

Initially considered lowercasing all text, but decided against it because:
- Capitalization affects sentence structure and noun usage
- Removing punctuation would interfere with Task 1

## Illustration Removal

Removed numerous instances of `[Illustration]` tags from Pride and Prejudice. These included:

**Example 1:**
```
[Illustration: M^{r.} & M^{rs.} Bennet

[_Copyright 1894 by George Allen._]]
```

**Example 2:**
```
[Illustration:

     "He rode a black horse"
]
```

**Regex used:** `\[Illustration:.*?\]\]?`

## Italics Removal

Now gutenberg uses underscore to write in italics which I will have to remove as it is not part of Jane Austens writing
so I will remove them by `text = re.sub(r"_([^_]+)_", r"\1", text)` 

# CHAPTER heading removal

I removed all chapter headings as AI generated won't have CHAPTER 1, 2 etc.

In [ ]:
import re
import unicodedata

with open("../data/processed/prideprejudice.txt", "r", encoding="utf-8") as f:
    text1 = f.read()

with open("../data/processed/cthulhu.txt", "r", encoding="utf-8") as f:
    text2 = f.read()

text1 = unicodedata.normalize("NFKC", text1)
text2 = unicodedata.normalize("NFKC", text2)

text1 = re.sub(r"\[Illustration:.*?\]\]?", "", text1, flags=re.DOTALL)
text1 = re.sub(r"_([^_]+)_", r"\1", text1)
text2 = re.sub(r"_([^_]+)_", r"\1", text2)
text1 = re.sub(r"\n\s*CHAPTER\s+[IVXLC0-9]+\.*\s*\n", "\n", text1, flags=re.IGNORECASE)
text2 = re.sub(r"\n\s*CHAPTER\s+[IVXLC0-9]+\.*\s*\n", "\n", text2, flags=re.IGNORECASE)

text1 = re.sub(r"\n+", " ", text1)
text1 = re.sub(r"\s+", " ", text1)
text1 = text1.strip()
text2 = re.sub(r"\n+", " ", text2)
text2 = re.sub(r"\s+", " ", text2)
text2 = text2.strip()



words1 = text1.split()
chunks1 = []
for i in range (0, len(words1), 160):
    chunk = words1[i:i+160]
    if len(chunk) >= 120:
        chunks1.append(" ".join(chunk))

words2 = text2.split()
chunks2 = []
for i in range (0, len(words2), 160):
    chunk = words2[i:i+160]
    if len(chunk) >= 120:
        chunks2.append(" ".join(chunk))

with open("../data/processed/prideprejudice.txt", "w", encoding="utf-8") as f:
    f.write(text1)

with open("../data/processed/cthulhu.txt", "w", encoding="utf-8") as f:
    f.write(text2)

with open("../data/processed/prideprejudicechunked.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(chunks1))

with open("../data/processed/cthulhuchunked.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(chunks2))


In [1]:
import google.generativeai as genai
import time
import os
from getpass import getpass

# Configure Gemini API
api_key = os.getenv("GEMINI_API_KEY")

# If not in environment, prompt the user to enter it
if not api_key:
    print("GEMINI_API_KEY not found in environment.")
    print("Paste your API key below (it will be hidden):")
    api_key = getpass("API Key: ")

if not api_key:
    raise ValueError("API key is required to continue.")

genai.configure(api_key=api_key)

# Read topics
with open("../data/processed/cthulhutopics.txt", "r") as f:
    cthulhu_topics = [line.strip() for line in f if line.strip()]

with open("../data/processed/prideprejudicetopics.txt", "r") as f:
    pride_prejudice_topics = [line.strip() for line in f if line.strip()]

print(f"Cthulhu topics: {len(cthulhu_topics)}")
print(f"Pride and Prejudice topics: {len(pride_prejudice_topics)}")
print("API configured successfully!")


GEMINI_API_KEY not found in environment.
Paste your API key below (it will be hidden):


/home/andaburger/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_7627/2715160079.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Cthulhu topics: 8
Pride and Prejudice topics: 9
API configured successfully!


In [ ]:
# Load chunked examples for few-shot prompting
with open("../data/processed/cthulhuchunked.txt", "r", encoding="utf-8") as f:
    cthulhu_examples = [p.strip() for p in f.read().split('\n') if p.strip()][:50]

with open("../data/processed/prideprejudicechunked.txt", "r", encoding="utf-8") as f:
    austen_examples = [p.strip() for p in f.read().split('\n') if p.strip()][:50]


import os
import time
import random
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import google.generativeai as genai

# Thread-safe lock for file writing and list appending
lock = threading.Lock()

# Class 2: Generate paragraphs - Generic style (no author mimicking)
def generate_class2_paragraphs(topics, output_file, num_paragraphs=500):
    """Generate paragraphs WITHOUT style constraint - baseline AI text (Multithreaded)"""
    paragraphs = []
    # Initialize model once (Google's GenAI client is generally thread-safe)
    model = genai.GenerativeModel('gemini-3-flash-preview')
    topics_str = ", ".join(topics)

    def _generate_one_class2(idx):
        try:
            prompt = f"""Write a single paragraph (100-200 words) about these topics: {topics_str}
                        Rules:
                        - Just write the paragraph, no introductions or meta-commentary
                        - Vary your sentence lengths naturally
                        - Don't be very complex, use relatively simple non nested sentences
                        """
            response = model.generate_content(prompt)
            text = response.text.strip()
            # Clean quotes if present
            if text.startswith('"') and text.endswith('"'):
                text = text[1:-1]
            return (idx, text)
        except Exception as e:
            print(f"Error generating paragraph {idx}: {e}")
            time.sleep(5)
            return None

    # Execute with 10 workers
    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(_generate_one_class2, i): i for i in range(num_paragraphs)}
        
        for future in as_completed(futures):
            result = future.result()
            if result is not None:
                with lock:
                    paragraphs.append(result)
                    count = len(paragraphs)
                    
                print(f"Generated {count}/{num_paragraphs} paragraphs")
                
                # Save progress every 50 paragraphs
                if count % 50 == 0:
                    with lock:
                        # Sort by index to maintain order roughly
                        sorted_p = [t for _, t in sorted(paragraphs, key=lambda x: x[0])]
                    
                    os.makedirs("../data/generated", exist_ok=True)
                    with open(output_file, "w", encoding="utf-8") as f:
                        f.write("\n\n".join(sorted_p))
                    print(f"✓ Saved {count} paragraphs to {output_file}")

    # Final save
    sorted_paragraphs = [t for _, t in sorted(paragraphs, key=lambda x: x[0])]
    os.makedirs("../data/generated", exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("\n\n".join(sorted_paragraphs))
    
    print(f"Saved {len(sorted_paragraphs)} paragraphs to {output_file}")


# Class 3: Jane Austen style generation
def generate_class3_austen_paragraphs(topics, output_file, examples, num_paragraphs, temperature=1.0):
    """Generate paragraphs mimicking Jane Austen's style (Multithreaded)"""
    paragraphs = []
    model = genai.GenerativeModel(
        'gemini-3-flash-preview',
        generation_config=genai.types.GenerationConfig(temperature=temperature)
    )
    topics_str = ", ".join(topics)

    def _generate_one_austen(idx):
        try:
            # Randomly select 3 examples for variety per thread call
            selected_examples = random.sample(examples, min(3, len(examples)))
            examples_text = "\n\n---\n\n".join(selected_examples)
            
            prompt = f"""You are an expert literary mimic specializing in Jane Austen's style. Study these authentic excerpts from Pride and Prejudice:

                        === AUTHENTIC EXAMPLES ===
                        {examples_text}
                        === END EXAMPLES ===

                        Now write a NEW paragraph (100-200 words) about: {topics_str}


                        Write a very readable and simple paragraph
                        DO not over describe
                        Don't use very complex or nested sentences, keep it relatively simple.
                        Try to have a high hapax legomana (many unique words) like Austen's style, but don't force it.
                        paragraph should start like it is from the pride and prejudice novel. 

                        JANE AUSTEN STYLE REQUIREMENTS:
                        1. Use some double quotes and dialogues and have a conversational tone at times, but don't overdo it
                        4. Use some(less) semicolons to connect related thoughts gracefully along with some(very less) em-dashes, some exclamations and questions as well like in a conversation(less), DO NOT OVERDO IT.
                        5. Maintain formal but conversational 19th-century English diction
                        6. Include characteristic Austen transitions: "however," "indeed," "moreover," "yet"
                        8. Keep readability at Flesch-Kincaid grade level ~8
                        9. Avoid excessive nesting - favor clarity over complexity

                        Output ONLY the paragraph. No explanations, no quotation marks, no meta-commentary."""

            response = model.generate_content(prompt)
            text = response.text.strip()
            if text.startswith('"') and text.endswith('"'):
                text = text[1:-1]
            return (idx, text)
        except Exception as e:
            print(f"Error generating Austen paragraph {idx}: {e}")
            time.sleep(5)
            return None

    # Execute with 10 workers
    with ThreadPoolExecutor(max_workers=20) as executor:
        futures = {executor.submit(_generate_one_austen, i): i for i in range(num_paragraphs)}
        
        for future in as_completed(futures):
            result = future.result()
            if result is not None:
                with lock:
                    paragraphs.append(result)
                    count = len(paragraphs)
                    
                print(f"Generated {count}/{num_paragraphs} paragraphs (temp={temperature})")
                
                # Save progress every 50 paragraphs
                if count % 50 == 0:
                    with lock:
                        sorted_p = [t for _, t in sorted(paragraphs, key=lambda x: x[0])]
                    
                    os.makedirs("../data/generated", exist_ok=True)
                    with open(output_file, "w", encoding="utf-8") as f:
                        f.write("\n\n".join(sorted_p))
                    print(f"✓ Saved {count} paragraphs to {output_file}")
    
    # Final save
    sorted_paragraphs = [t for _, t in sorted(paragraphs, key=lambda x: x[0])]
    os.makedirs("../data/generated", exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("\n\n".join(sorted_paragraphs))
    
    print(f"✓ Final save: {len(sorted_paragraphs)} paragraphs to {output_file}")


# Class 3: H.P. Lovecraft style generation
def generate_class3_lovecraft_paragraphs(topics, output_file, examples, num_paragraphs, temperature=1.0):
    """Generate paragraphs mimicking H.P. Lovecraft's style (Multithreaded)"""
    paragraphs = []
    model = genai.GenerativeModel(
        'gemini-3-flash-preview',
        generation_config=genai.types.GenerationConfig(temperature=temperature)
    )
    topics_str = ", ".join(topics)

    def _generate_one_lovecraft(idx):
        try:
            # Randomly select 4 examples for variety per thread call
            selected_examples = random.sample(examples, min(4, len(examples)))
            examples_text = "\n\n---\n\n".join(selected_examples)
            
            prompt = f"""You are an expert literary mimic specializing in H.P. Lovecraft's style. Study these authentic excerpts from The Call of Cthulhu:

=== AUTHENTIC EXAMPLES ===
{examples_text}
=== END EXAMPLES ===

Now write a NEW paragraph (100-200 words) about: {topics_str}

Write a very very simple and readable paragraph, nothing complicated, have very less syntactic complexity
DO not over describe
Don't use very complex or nested sentences, keep it relatively simple.
have a high hapax legomana.
H.P. LOVECRAFT STYLE REQUIREMENTS:
1. Build mounting dread and cosmic horror through accumulating details
4. Use some(less) semicolons to connect related thoughts gracefully along with some(very less) em-dashes, some exclamations and questions as well like in a conversation(less), DO NOT OVERDO IT.
3. Keep readability at Flesch-Kincaid grade level ~8 (don't overdo complexity)

Output ONLY the paragraph. No explanations, no quotation marks, no meta-commentary."""

            response = model.generate_content(prompt)
            text = response.text.strip()
            if text.startswith('"') and text.endswith('"'):
                text = text[1:-1]
            return (idx, text)
        except Exception as e:
            print(f"Error generating Lovecraft paragraph {idx}: {e}")
            time.sleep(5)
            return None

    # Execute with 10 workers
    with ThreadPoolExecutor(max_workers=20) as executor:
        futures = {executor.submit(_generate_one_lovecraft, i): i for i in range(num_paragraphs)}
        
        for future in as_completed(futures):
            result = future.result()
            if result is not None:
                with lock:
                    paragraphs.append(result)
                    count = len(paragraphs)
                    
                print(f"Generated {count}/{num_paragraphs} paragraphs (temp={temperature})")
                
                # Save progress every 50 paragraphs
                if count % 50 == 0:
                    with lock:
                        sorted_p = [t for _, t in sorted(paragraphs, key=lambda x: x[0])]
                    
                    os.makedirs("../data/generated", exist_ok=True)
                    with open(output_file, "w", encoding="utf-8") as f:
                        f.write("\n\n".join(sorted_p))
                    print(f"✓ Saved {count} paragraphs to {output_file}")
    
    # Final save
    sorted_paragraphs = [t for _, t in sorted(paragraphs, key=lambda x: x[0])]
    os.makedirs("../data/generated", exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("\n\n".join(sorted_paragraphs))
    
    print(f"✓ Final save: {len(sorted_paragraphs)} paragraphs to {output_file}")

In [ ]:
# Generate Class 2 paragraphs (generic style - NO author mimicking)
print("Generating Class 2 paragraphs for Cthulhu topics...")
generate_class2_paragraphs(cthulhu_topics, "../data/generated/cthulhu_class2.txt", num_paragraphs=500)

print("\nGenerating Class 2 paragraphs for Pride and Prejudice topics...")
generate_class2_paragraphs(pride_prejudice_topics, "../data/generated/prideprejudice_class2.txt", num_paragraphs=500)

# Generate Class 3 paragraphs with author-specific functions
print("\nGenerating Class 3 paragraphs for Cthulhu topics (H.P. Lovecraft style)...")
generate_class3_lovecraft_paragraphs(
    cthulhu_topics, 
    "../data/generated/cthulhu_class3.txt", 
    examples=cthulhu_examples,
    num_paragraphs=500,
    temperature=0.8  # Adjust this value to experiment (0.0-2.0)
)

print("\nGenerating Class 3 paragraphs for Pride and Prejudice topics (Jane Austen style)...")
generate_class3_austen_paragraphs(
    pride_prejudice_topics, 
    "../data/generated/prideprejudice_class3.txt", 
    examples=austen_examples,
    num_paragraphs=500,
    temperature=0.8  # Adjust this value to experiment (0.0-2.0)
)

print("\nAll paragraphs generated successfully!")


Generating Class 2 paragraphs for Cthulhu topics...
Generated 1/500 paragraphs
Generated 2/500 paragraphs
Generated 3/500 paragraphs
Generated 4/500 paragraphs
Generated 5/500 paragraphs
Generated 6/500 paragraphs
Generated 7/500 paragraphs
Generated 8/500 paragraphs
Generated 9/500 paragraphs
Generated 10/500 paragraphs
Generated 11/500 paragraphs
Generated 12/500 paragraphs
Generated 13/500 paragraphs
Generated 14/500 paragraphs
Generated 15/500 paragraphs
Generated 16/500 paragraphs
Generated 17/500 paragraphs
Generated 18/500 paragraphs
Generated 19/500 paragraphs
Generated 20/500 paragraphs
Generated 21/500 paragraphs
Generated 22/500 paragraphs
Generated 23/500 paragraphs
Generated 24/500 paragraphs
Generated 25/500 paragraphs
Generated 26/500 paragraphs
Generated 27/500 paragraphs
Generated 28/500 paragraphs
Generated 29/500 paragraphs
Generated 30/500 paragraphs
Generated 31/500 paragraphs
Generated 32/500 paragraphs
Generated 33/500 paragraphs
Generated 34/500 paragraphs
Gener

In [3]:
# Experiment with different temperatures for Jane Austen style
# Temperature guide:
# - 0.0-0.3: Very deterministic, conservative (may be too rigid)
# - 0.4-0.7: Balanced creativity and consistency (recommended range)
# - 0.8-1.2: More creative and varied (default)
# - 1.3-2.0: Highly creative but may deviate from style

print("\nGenerating Class 3 paragraphs for Cthulhu topics (H.P. Lovecraft style)...")
generate_class3_lovecraft_paragraphs(
    cthulhu_topics, 
    "../data/generated/cthulhu_class3.txt", 
    examples=cthulhu_examples,
    num_paragraphs=500,
    temperature=1.0  # Adjust this value to experiment (0.0-2.0)
)

print("Generating Class 3 paragraphs for Pride and Prejudice topics (Jane Austen style)...")
generate_class3_austen_paragraphs(
    pride_prejudice_topics, 
    "../data/generated/prideprejudice_class3.txt", 
    examples=austen_examples,
    num_paragraphs=500,
    temperature=1.0  # Experiment with values: try 0.5, 0.8, 1.0, 1.2
)

print("\nClass 3 paragraphs generated successfully!")



Generating Class 3 paragraphs for Cthulhu topics (H.P. Lovecraft style)...
Generated 1/500 paragraphs (temp=1.0)
Generated 2/500 paragraphs (temp=1.0)
Generated 3/500 paragraphs (temp=1.0)
Generated 4/500 paragraphs (temp=1.0)
Generated 5/500 paragraphs (temp=1.0)
Generated 6/500 paragraphs (temp=1.0)
Generated 7/500 paragraphs (temp=1.0)
Generated 8/500 paragraphs (temp=1.0)
Generated 9/500 paragraphs (temp=1.0)
Generated 10/500 paragraphs (temp=1.0)
Generated 11/500 paragraphs (temp=1.0)
Generated 12/500 paragraphs (temp=1.0)
Generated 13/500 paragraphs (temp=1.0)
Generated 14/500 paragraphs (temp=1.0)
Generated 15/500 paragraphs (temp=1.0)
Generated 16/500 paragraphs (temp=1.0)
Generated 17/500 paragraphs (temp=1.0)
Generated 18/500 paragraphs (temp=1.0)
Generated 19/500 paragraphs (temp=1.0)
Generated 20/500 paragraphs (temp=1.0)
Generated 21/500 paragraphs (temp=1.0)
Generated 22/500 paragraphs (temp=1.0)
Generated 23/500 paragraphs (temp=1.0)
Generated 24/500 paragraphs (temp=1.